In [3]:
import mlflow
import os
import numpy as np
import pandas as pd
import psycopg

In [7]:
connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postres_credentials = {
    "host" : os.environ.get("DB_DESTINATION_HOST"),
    "port" : os.environ.get("DB_DESTINATION_PORT"),
    "dbname" : os.environ.get("DB_DESTINATION_NAME"),
    "user" : os.environ.get("DB_DESTINATION_USER"),
    "password" : os.environ.get("DB_DESTINATION_PASSWORD"),
}

assert all([var_val != "" for var_val in list(postres_credentials.values())])

connection.update(postres_credentials)

In [9]:
TABLE_NAME = "clean_realestate_data"

In [10]:
with psycopg.connect(**connection) as conn:
    pass

    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")
        
        data = cur.fetchall()

        columns = [col[0] for col in cur.description]

In [11]:
df = pd.DataFrame(data, columns=columns)

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [17]:
X_tr, X_val, y_tr, y_val = train_test_split(df, df["price"])

In [14]:
num_features = df.select_dtypes(["float", "int"])
preprocess = ColumnTransformer([
        ("nums", StandardScaler(), num_features.columns.tolist())
    ],
    remainder="drop",
    verbose_feature_names_out=False
    )

In [15]:
model = LinearRegression()
pipeline = Pipeline([
        ("preprocess", preprocess),
        ("model", model)
    ])

In [18]:
pipeline.fit(X_tr, y_tr)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('nums', StandardScaler(),
                                                  ['id', 'build_year',
                                                   'building_type_int',
                                                   'latitude', 'longitude',
                                                   'ceiling_height',
                                                   'flats_count',
                                                   'floors_total',
                                                   'has_elevator', 'floor',
                                                   'is_apartment',
                                                   'kitchen_area',
                                                   'living_area', 'rooms',
                                                   'studio', 'total_area',
                                                   'price'])],
                                   verbose_feature_names_out=False)),
                ('model', LinearRegression())])

In [19]:
prediction = pipeline.predict(X_val)

In [20]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import log_loss

In [21]:
cv_strategy = KFold(n_splits=5, shuffle=True)

In [16]:
cv_res = cross_validate(model, X_val, y_val, cv=cv_strategy, n_jobs=-1, scoring=['neg_mean_squared_log_error', 'r2'])

In [17]:
for k, v in cv_res.items():
    cv_res[k] = round(v.mean(), 3)

In [1]:
EXPERIMENT_NAME = "MY_FIRST_EXPERIMENT"
RUN_NAME = "model_0_registered"
REGISTRY_MODEL_NAME = "REAL_ESTATE"

In [32]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_registry_uri("http://127.0.0.1:5000")

In [34]:
pip_requirements = "./requirements.txt"
signature = mlflow.models.infer_signature(
	X_val,
	prediction
)
input_example = X_val[:10]
metadata = {"model_type": "monthly"}

In [35]:
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment_id:
    experiment_id = experiment_id.experiment_id
else:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

In [37]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["AWS_ACCESS_KEY_ID"] = os.environ.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.environ.get("AWS_SECRET_ACCESS_KEY")

In [38]:
with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    model_info = mlflow.sklearn.log_model(sk_model=model,
                                          artifact_path="models",
                                          registered_model_name=REGISTRY_MODEL_NAME,
                                          await_registration_for=60,
                                          pip_requirements=pip_requirements,
                                          signature=signature,
                                          input_example=input_example,
                                          metadata=metadata)

Successfully registered model 'REAL_ESTATE'.
2026/01/03 09:59:43 INFO mlflow.tracking._model_registry.client: Waiting up to 60 seconds for model version to finish creation. Model name: REAL_ESTATE, version 1
Created version '1' of model 'REAL_ESTATE'.


In [39]:
loaded_model = mlflow.sklearn.load_model(model_uri=model_info.model_uri)

In [40]:
model_predict = loaded_model.predict(X_val)

/home/mle-user/mle_projects/mlflow/.venv/lib/python3.10/site-packages/sklearn/base.py:458: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(


In [42]:
X_val

,id,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator,floor,is_apartment,kitchen_area,living_area,rooms,studio,total_area,price
108821,32364,1972,6,55.600410,37.653530,2.64,98,16,1,10,0,0.0,0.000000,2,0,40.000000,7950000.0
116709,41172,1983,4,55.583515,37.674850,2.64,427,12,1,1,0,8.7,0.000000,1,0,38.799999,7750000.0
31443,74696,1977,4,55.620888,37.750854,2.50,191,12,1,9,0,6.6,44.000000,3,0,65.000000,13100000.0
37563,81763,1984,1,55.587536,37.668930,2.70,112,14,1,11,0,9.0,32.000000,2,0,56.000000,10800000.0
94727,15996,1968,6,55.780087,37.731873,2.64,178,12,1,7,0,10.0,19.600000,1,0,36.000000,6800000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71910,124919,2014,2,55.589146,37.456398,2.80,256,5,1,4,0,9.4,15.000000,1,0,35.599998,9900000.0
87081,868,1970,4,55.622204,37.593990,2.64,288,9,1,6,0,6.0,43.560001,4,0,63.880001,13800000.0
44457,90494,2007,4,55.718540,37.746769,2.64,496,23,1,15,0,10.0,32.000000,2,0,61.700001,15500000.0
92420,13012,1970,4,55.621868,37.596867,2.64,216,9,1,5,0,3.0,30.000000,2,0,45.000000,11200000.0


In [41]:
model_predict

array([3.69493597e+13, 3.60198164e+13, 6.08851007e+13, ...,
       7.20396208e+13, 5.20544390e+13, 6.16891557e+13])

In [43]:
model_predict.dtype

dtype('float64')